# Phase 1.3 : Architecture CNN from scratchChats vs chiens (Microsoft Cats vs Dogs Dataset). Suite de `phase1_2_preprocessing.ipynb`.**Ce notebook est autonome** : la section "Reprise" ci-dessous refait le setup des phases 1.1 et 1.2 (téléchargement, organisation, pipeline de données), sans les explications déjà vues.

## Reprise (setup phases 1.1 + 1.2, condensé)Nécessite un `kaggle.json` (voir phase 1.1 pour l'obtenir).

In [ ]:
CLASS_A = "cat"CLASS_B = "dog"DATA_ROOT = "/content/data"

In [ ]:
!pip install kaggle -q!mkdir -p ~/.kaggle

In [ ]:
from google.colab import filesfiles.upload()  # sélectionner kaggle.jsonimport osos.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

In [ ]:
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p /content/data/!cd /content/data && unzip -q microsoft-catsvsdogs-dataset.zip

In [ ]:
import os, shutil, randomfrom PIL import ImageRAW_DIR = os.path.join(DATA_ROOT, "PetImages")raw_dirs = {CLASS_A: os.path.join(RAW_DIR, "Cat"), CLASS_B: os.path.join(RAW_DIR, "Dog")}def list_valid_images(folder):    valid = []    for fname in os.listdir(folder):        path = os.path.join(folder, fname)        if os.path.getsize(path) == 0:            continue        try:            with Image.open(path) as img:                img.verify()            valid.append(path)        except Exception:            continue    return validfiles_a = list_valid_images(raw_dirs[CLASS_A])files_b = list_valid_images(raw_dirs[CLASS_B])for split in ["train", "val"]:    for cls in [CLASS_A, CLASS_B]:        os.makedirs(os.path.join(DATA_ROOT, split, cls), exist_ok=True)random.seed(42)random.shuffle(files_a)random.shuffle(files_b)def split_and_copy(file_list, cls):    cut = int(len(file_list) * 0.8)    train_files, val_files = file_list[:cut], file_list[cut:]    for f in train_files:        shutil.copy(f, os.path.join(DATA_ROOT, "train", cls, os.path.basename(f)))    for f in val_files:        shutil.copy(f, os.path.join(DATA_ROOT, "val", cls, os.path.basename(f)))split_and_copy(files_a, CLASS_A)split_and_copy(files_b, CLASS_B)print(f"{CLASS_A}: {len(files_a)} images valides")print(f"{CLASS_B}: {len(files_b)} images valides")

In [ ]:
import tensorflow as tfIMG_SIZE = (128, 128)BATCH_SIZE = 32train_ds = tf.keras.utils.image_dataset_from_directory(    os.path.join(DATA_ROOT, "train"),    labels="inferred",    label_mode="binary",    class_names=[CLASS_A, CLASS_B],    color_mode="rgb",    image_size=IMG_SIZE,    batch_size=BATCH_SIZE,    shuffle=True,    seed=42,)val_ds = tf.keras.utils.image_dataset_from_directory(    os.path.join(DATA_ROOT, "val"),    labels="inferred",    label_mode="binary",    class_names=[CLASS_A, CLASS_B],    color_mode="rgb",    image_size=IMG_SIZE,    batch_size=BATCH_SIZE,    shuffle=False,    seed=42,)normalization_layer = tf.keras.layers.Rescaling(1./255)train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))AUTOTUNE = tf.data.AUTOTUNEtrain_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Phase 1.3 : Architecture CNN from scratch**Objectif** : construire et compiler un CNN from scratch, vérifier que les shapes du `model.summary()` correspondent à celles calculées à la main, et identifier le nombre de paramètres.Avec `IMG_SIZE = (128, 128)`, les shapes attendues (identiques à l'exemple du cours) :| Couche | Shape sortie ||---|---|| Input | 128 x 128 x 3 || Conv2D(32) + MaxPool | 64 x 64 x 32 || Conv2D(64) + MaxPool | 32 x 32 x 64 || Conv2D(128) + MaxPool | 16 x 16 x 128 || Flatten | 32 768 || Dense(128) | 128 || Dense(1) | 1 |

In [ ]:
from tensorflow.keras import layers, modelsdef build_cnn_scratch(input_shape):    """CNN from scratch pour la classification binaire.    Architecture délibérément simple : on veut voir l'overfitting, pas l'éviter.    """    model = models.Sequential([        layers.Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=input_shape),        layers.MaxPooling2D((2, 2)),        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),        layers.MaxPooling2D((2, 2)),        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),        layers.MaxPooling2D((2, 2)),        layers.Flatten(),        layers.Dense(128, activation="relu"),        layers.Dense(1, activation="sigmoid"),    ])    return modelmodel_scratch = build_cnn_scratch(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))model_scratch.summary()

In [ ]:
model_scratch.compile(    optimizer="adam",    loss="binary_crossentropy",    metrics=["accuracy"],)

### Vérifications avant la phase 1.4**Questions avant d'exécuter** (à comparer avec le `model.summary()` une fois lancé) :- Shape en sortie du bloc 3 (après le 3e MaxPooling) pour `IMG_SIZE=(128,128)` : `16 x 16 x 128`.- Nombre d'éléments après `Flatten` : `16 * 16 * 128 = 32 768`.- Paramètres de `Dense(128)` : `32 768 * 128 + 128 = 4 194 432` — la couche la plus lourde du réseau (~97% du total).**Quality gate** :- **Happy path** : `model.summary()` affiche 3 blocs Conv2D + Flatten + 2 Dense, output shape finale `(None, 1)`, total params ≈ 4 287 809 (~4.3M) — identique au calcul ci-dessus.- **Edge case** : remplacer `padding="same"` par `padding="valid"` sur les Conv2D. Les shapes intermédiaires se réduisent (chaque Conv perd 2 pixels par bord), `Flatten` produit moins d'éléments, `Dense(128)` a moins de paramètres.- **Adversarial** : oublier `activation="sigmoid"` sur la dernière `Dense(1)`. Le modèle compile sans erreur mais la `binary_crossentropy` reçoit des logits non bornés — l'entraînement diverge ou plafonne à 50%. Réflexe : toujours vérifier la dernière activation avant `fit()`.